In [14]:
!pip install "accelerate>=1.1.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Phase 4: Trainer with weighted loss

In [2]:
import json
import torch
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_ds = load_from_disk("data/train_dataset")
val_ds   = load_from_disk("data/val_dataset")
test_ds  = load_from_disk("data/test_dataset")

print(train_ds, val_ds, test_ds)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 2248760
}) Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 281095
}) Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 281103
})


In [4]:
with open("data/class_weights.json") as f:
    class_weights = json.load(f)


In [5]:
labels = sorted(class_weights.keys())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

In [6]:
weights_tensor = torch.tensor(
    [class_weights[id2label[i]] for i in range(len(id2label))],
    dtype=torch.float
)

In [7]:
MODEL_NAME = "markusbayer/CySecBERT" # replace with CysecBERT if needed

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 718.96it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: markusbayer/CySecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the check

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # not needed if already tokenized

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

In [9]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights_tensor.to(logits.device)
        )

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [10]:
training_args = TrainingArguments(
    output_dir="outputs",
    eval_strategy="steps",
    eval_steps=2000,
    logging_steps=500,
    save_steps=2000,
    save_total_limit=2,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    learning_rate=2e-5,
    num_train_epochs=1,   # sanity check only
    fp16=torch.cuda.is_available(),

    report_to="none",
    load_best_model_at_end=False,

    remove_unused_columns=True
)


In [11]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

In [12]:

def on_the_fly_transform(batch):
    # This runs only on the small batches requested during training
    batch["labels"] = [label2id[label] for label in batch["label"]]
    # Remove the original string label so the collator doesn't try to tensorize it!
    if "label" in batch:
        del batch["label"]
        
    return batch

# Apply transform (uses NO disk space)
train_ds.set_transform(on_the_fly_transform)
val_ds.set_transform(on_the_fly_transform)
test_ds.set_transform(on_the_fly_transform)


In [13]:

# Create a tiny dataloader with padding
sanity_loader = DataLoader(
    train_ds.select(range(8)),
    batch_size=8,
    collate_fn=data_collator
)

batch = next(iter(sanity_loader))

out = model(**batch)

print("Logits shape:", out.logits.shape)

Logits shape: torch.Size([8, 15])


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
